# Module 02: Pandas for Machine Learning
## Notebook 01: Series and DataFrame Fundamentals

Pandas is the primary data manipulation and exploratory analysis library in the Python data science stack. While NumPy provides the raw numeric compute engine, Pandas provides the labeled, multi-type tabular structures required for real-world machine learning datasets.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Construct and inspect 1D `pd.Series` and 2D `pd.DataFrame` structures.
2. Load tabular data from CSV and JSON formats with custom parsing parameters.
3. Perform initial exploratory data audits using `.info()`, `.describe()`, and `.shape`.
4. Inspect and convert data types (`dtypes`).
5. Drastically reduce memory footprints using category conversion and numeric downcasting.

In [2]:
import pandas as pd
import numpy as np
import io

print(f"Pandas version: {pd.__version__}")

Pandas version: 3.0.6


### 1. The 1D Building Block: `pd.Series`

A `Series` is a 1-dimensional labeled array capable of holding any data type.
- Unlike a 1D NumPy array, a `Series` possesses an explicit **index** (row labels).
- Operations align automatically based on the index labels.

In [3]:
# Creating a Series from a Python list with custom index
temperatures = pd.Series([22.5, 24.0, 19.8, 25.2], index=['London', 'Paris', 'Berlin', 'Madrid'], name="Temperature_C")

print(temperatures)
print("\nIndex:  ", temperatures.index)
print("Values: ", temperatures.values)
print("Access by label ('Berlin'):", temperatures['Berlin'])
print("Vectorized conversion (Fahrenheit):\n", temperatures * 9/5 + 32)

London    22.5
Paris     24.0
Berlin    19.8
Madrid    25.2
Name: Temperature_C, dtype: float64

Index:   Index(['London', 'Paris', 'Berlin', 'Madrid'], dtype='str')
Values:  [22.5 24.  19.8 25.2]
Access by label ('Berlin'): 19.8
Vectorized conversion (Fahrenheit):
 London    72.50
Paris     75.20
Berlin    67.64
Madrid    77.36
Name: Temperature_C, dtype: float64


---
### 2. The 2D Tabular Engine: `pd.DataFrame`

A `DataFrame` represents a 2D tabular dataset where:
- Rows are indexed observations (samples $N$).
- Columns are named variables/features (dimensions $D$).
- Each column is internally an individual `pd.Series`.

In [4]:
# Creating a DataFrame from a dictionary of lists
data_dict = {
    'Customer_ID': [1001, 1002, 1003, 1004, 1005],
    'Age': [28, 45, 33, 54, 23],
    'Annual_Income': [55000.0, 85000.0, 62000.0, 110000.0, 42000.0],
    'Loyalty_Member': [True, True, False, True, False]
}

df_customers = pd.DataFrame(data_dict)
print("Customer DataFrame:\n", df_customers)
print("\nDataFrame Shape (Samples, Features):", df_customers.shape)

Customer DataFrame:
    Customer_ID  Age  Annual_Income  Loyalty_Member
0         1001   28        55000.0            True
1         1002   45        85000.0            True
2         1003   33        62000.0           False
3         1004   54       110000.0            True
4         1005   23        42000.0           False

DataFrame Shape (Samples, Features): (5, 4)


---
### 3. Loading Tabular Data

In machine learning projects, data usually originates from CSV, Parquet, or JSON files.
Let's simulate loading an incoming CSV dataset using `pd.read_csv()`.

In [5]:
raw_csv_data = '''customer_id,churn,monthly_charges,total_charges,contract_type,tenure_months
C-01,No,29.85,29.85,Month-to-month,1
C-02,Yes,56.95,1889.50,One year,34
C-03,No,53.85,108.15,Month-to-month,2
C-04,No,42.30,1840.75,One year,45
C-05,Yes,70.70,151.65,Month-to-month,2
C-06,No,89.10,267.30,Two year,3
C-07,No,29.75,301.90,Month-to-month,10
'''

# Read CSV with Pandas
df = pd.read_csv(io.StringIO(raw_csv_data))
print("Loaded DataFrame (first 3 rows):\n", df.head(3))

Loaded DataFrame (first 3 rows):
   customer_id churn  monthly_charges  total_charges   contract_type  \
0        C-01    No            29.85          29.85  Month-to-month   
1        C-02   Yes            56.95        1889.50        One year   
2        C-03    No            53.85         108.15  Month-to-month   

   tenure_months  
0              1  
1             34  
2              2  


---
### 4. Exploratory Data Auditing: `head`, `info`, and `describe`

When receiving a new machine learning dataset, standard exploratory audits include:
1. `df.head(n)` / `df.tail(n)`: Inspect actual samples.
2. `df.info()`: Check memory usage, column names, and non-null counts.
3. `df.describe()`: Summary statistics for numeric and categorical features.

In [6]:
print("=== DataFrame Info ===")
df.info()

print("\n=== Numerical Feature Summary Statistics ===")
print(df.describe().round(2))

print("\n=== Categorical Summary Statistics ===")
print(df.describe(include=['object']))

=== DataFrame Info ===
<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      7 non-null      str    
 1   churn            7 non-null      str    
 2   monthly_charges  7 non-null      float64
 3   total_charges    7 non-null      float64
 4   contract_type    7 non-null      str    
 5   tenure_months    7 non-null      int64  
dtypes: float64(2), int64(1), str(3)
memory usage: 468.0 bytes

=== Numerical Feature Summary Statistics ===
       monthly_charges  total_charges  tenure_months
count             7.00           7.00           7.00
mean             53.21         655.59          13.86
std              21.69         831.48          18.05
min              29.75          29.85           1.00
25%              36.08         129.90           2.00
50%              53.85         267.30           3.00
75%              63.82        1071.32   

/tmp/ipykernel_64966/1150067664.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.describe(include=['object']))


---
### 5. Memory Optimization for Large-Scale Datasets

Default Pandas imports often load strings as `object` (pointer overhead) and numbers as 64-bit (`int64`, `float64`).
For datasets with millions of rows:
- Convert repetitive string columns with low cardinality to `category` dtype.
- Downcast numeric columns using `pd.to_numeric(..., downcast='integer'|'float')`.

In [7]:
initial_mem = df.memory_usage(deep=True).sum()

# 1. Convert categorical strings to 'category'
df['contract_type'] = df['contract_type'].astype('category')
df['churn'] = df['churn'].astype('category')

# 2. Downcast numeric columns
df['tenure_months'] = pd.to_numeric(df['tenure_months'], downcast='integer')
df['monthly_charges'] = pd.to_numeric(df['monthly_charges'], downcast='float')

optimized_mem = df.memory_usage(deep=True).sum()

print(f"Initial Memory:   {initial_mem} bytes")
print(f"Optimized Memory: {optimized_mem} bytes")
print(f"Memory reduction: {((initial_mem - optimized_mem) / initial_mem) * 100:.1f}%")
print("\nOptimized dtypes:\n", df.dtypes)

Initial Memory:   1453 bytes
Optimized Memory: 888 bytes
Memory reduction: 38.9%

Optimized dtypes:
 customer_id             str
churn              category
monthly_charges     float32
total_charges       float64
contract_type      category
tenure_months          int8
dtype: object


### Summary & Next Steps
In this notebook, you mastered:
- Properties of `pd.Series` and `pd.DataFrame`.
- Reading tabular data and executing exploratory schema audits.
- Categorical and numeric memory downcasting techniques.

**Next Notebook:** `02_indexing_filtering_and_assignment.ipynb` — Master `.loc` vs `.iloc`, compound boolean filtering, `.query()`, and avoiding `SettingWithCopyWarning`.